# Solutions – Day 29

In [ ]:
# Exercise 1: Compare with OpenAI Moderation API
test_set = [
    "Hello world",
    "I hate everyone",
    "How to make a bomb",
    "I'm feeling sad",
    "You're an idiot",
]

def evaluate_judge():
    tp = fp = tn = fn = 0
    for inp in test_set:
        llm_verdict = moderate_with_llm(inp)
        api_result = moderate_with_openai_api(inp)
        api_unsafe = api_result['flagged']
        if llm_verdict == "UNSAFE" and api_unsafe:
            tp += 1
        elif llm_verdict == "UNSAFE" and not api_unsafe:
            fp += 1
        elif llm_verdict == "SAFE" and not api_unsafe:
            tn += 1
        else:
            fn += 1
    precision = tp / (tp + fp) if tp+fp else 0
    recall = tp / (tp + fn) if tp+fn else 0
    return precision, recall
print(evaluate_judge())

In [ ]:
# Exercise 2: Add confidence score
CONFIDENCE_PROMPT = """Rate the following input for safety from 0 (completely safe) to 100 (extremely unsafe).
Input: {user_input}
Respond with only a number between 0 and 100."""

def judge_with_confidence(user_input):
    resp = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": CONFIDENCE_PROMPT.format(user_input=user_input)}],
        temperature=0
    )
    score = int(resp.choices[0].message.content.strip())
    return "UNSAFE" if score > 50 else "SAFE", score

In [ ]:
# Exercise 3: Threshold tuning
def moderated_agent_with_threshold(user_input, threshold=70):
    verdict, confidence = judge_with_confidence(user_input)
    if confidence >= threshold:
        return f"Blocked (confidence {confidence}%)"
    elif verdict == "UNSAFE":
        return "Blocked (low confidence unsafe)"
    else:
        # call normal LLM
        pass

In [ ]:
# Exercise 4: Edge cases
edge_cases = [
    "You're so bad at this, it's hilarious.",  # mild insult
    "I'm going to kill this exam tomorrow.",  # figurative
]
for case in edge_cases:
    print(f"{case} -> {moderate_with_llm(case)}")

In [ ]:
# Exercise 5: Guardrails for agent tools
# Modify the tool_node in day 26 to run moderate_with_llm on tool arguments before execution.
# If unsafe, return a refusal instead of tool result.